In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

data=pd.read_csv("data/sp500_2015_2024_clean_tuning.csv", parse_dates=["Date"], index_col="Date")

feature_cols=["Close", "Returns", "MA10", "MA20", "MA50", "MA100", "Close_lag1", "Close_lag5", "Volume", "Vol20"]
X=data[feature_cols]

y=(data["Close_tomorrow"]/data["Close"])-1

split_ratio = 0.6
split_index = int(len(data) * split_ratio)

X_train_raw, X_test_raw = X.iloc[:split_index], X.iloc[split_index:]
y_train_raw, y_test_raw = y.iloc[:split_index], y.iloc[split_index:]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

X_train_scaled = pd.DataFrame(X_train_scaled, index = X_train_raw.index, columns = feature_cols)
X_test_scaled = pd.DataFrame(X_test_scaled, index = X_test_raw.index, columns = feature_cols)

def make_sequences(X_df, y_series, lookback = 30):
    X_list, y_list, idx_list = [], [], []
    for i in range(lookback, len(X_df)):
        X_list.append(X_df.iloc[i-lookback:i].values)
        y_list.append(y_series.iloc[i])
        idx_list.append(y_series.index[i])
        return np.array(X_list), np.array(y_list), np.array(idx_list)

lookback = 30

X_train_seq, y_train_seq, idx_train = make_sequences(X_train_scaled, y_train_raw, lookback)
X_test_seq, y_test_seq, idx_test = make_sequences(X_test_scaled, y_test_raw, lookback)

X_train_seq.shape, X_test_seq.shape

In [ ]:
model = Sequential([ 
    LSTM(50, return_sequences=True, input_shape=(30, 10)), 
    Dropout(0.2), 
    LSTM(50, return_sequences = False), 
    Dropout(0.2), 
    Dense(25), 
    Dense(1) ])

model.compile(optimizer='adam', loss='mse')
print("Model created")

history = model.fit(X_train_seq, y_train, epochs=20, batch_size=1,
                    validation_data=(X_test_seq, y_test), verbose=1)

print("LSTM training complete")
print(f"Final Loss: {history.history['loss'][-1]:.6f}")

In [ ]:
y_pred = model.predict(X_test_seq).flatten()
mse = np.mean((y_test - y_pred.flatten())**2)
print(f"Test MSE : {mse:.6f}")

test_df=pd.DataFrame({'actual':y_test_seq, 'pred':y_pred}, index=idx_test)
test_df.to_csv('lstm_predictions.csv')

In [ ]:
test_df['signal'] = np.where(test_df['pred'] > test_df['actual'] *1.002, 1, 0)
test_df['returns'] = test_df['actual'].pct_change()
test_df['strat_ret'] = test_df['signal'].shift(1)*test_df['returns']
bh_ret = ((test_df['actual'].iloc[-1]/test_df['actual'].iloc[0])-1) * 100
strat_ret = (1+test_df['strat_ret'].dropna()).prod() - 1
strat_ret *= 100
print(f"B&H: {bh_ret:.1f}% | Strat: {strat_ret:.1f}%")

import matplotlib.pyplot as plt
plt.figure(figsize=(12, 5))
pd.Series((1+test_df['returns']).cumprod(), index=test_df.index).plot(label = "Buy-Hold")
pd.Series((1+test_df['strat_ret']).cumprod(), index=test_df.index).plot(label = "LSTM strt")
plt.legend()
plt.title("LSTM Backtest")
plt.savefig("lstm_bacltest.png")



In [ ]:
print(test_df[['actual','returns','signal','strat_ret']].head(10))
print(test_df[['actual','returns','signal','strat_ret']].tail(10))
print(test_df['returns'].describe())
print(test_df['strat_ret'].describe())
print(len(test_df))
